# DREAM AI — train our native model from random initialization

This notebook trains DREAM's own PyTorch transformer. It does **not** download or call another model, use a third-party AI API, or import pretrained weights. The checkpoint starts with random weights.

**Important:** the current architecture is an early research-scale model, not a capable general assistant. Training requires a substantial, high-quality dataset that you have permission to use. Do not mark rights/privacy/quality review as complete unless a person has actually completed those checks. Free GPU availability and session duration are not guaranteed.

In [ ]:
sft_completed = False
from pathlib import Path
import os, subprocess, sys, torch

REPO = Path('/content/dream-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mjnr0477/dream-ai.git', str(REPO)], check=True)
os.chdir(REPO)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. You can run a tiny smoke test on CPU, but substantial training will be slow.')

## 1. Upload the reviewed DREAM corpus

Prepare a JSONL file named `approved-records.jsonl`. Each line must contain `id`, `source`, `source_group`, `license`, `rights_confirmed`, `privacy_reviewed`, `quality_reviewed`, `language`, and `text`. The three review fields must be `true` only after real human review. Use at least three independent source groups. Keep personal or confidential information out of the corpus.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'approved-records.jsonl' not in uploaded:
    raise RuntimeError('Please upload a file named approved-records.jsonl')
Path('data').mkdir(exist_ok=True)
Path('data/approved-records.jsonl').write_bytes(uploaded['approved-records.jsonl'])
print('Uploaded bytes:', Path('data/approved-records.jsonl').stat().st_size)

In [ ]:
subprocess.run([sys.executable, 'model/prepare_corpus.py', '--input', 'data/approved-records.jsonl', '--output', 'data/prepared'], check=True)
import json
manifest = json.loads(Path('data/prepared/manifest.json').read_text())
print(json.dumps(manifest, indent=2))
for split in ('train', 'validation', 'test'):
    size = Path(f'data/prepared/{split}.txt').stat().st_size
    print(split, 'UTF-8 bytes:', size)
    if size < 258:
        raise RuntimeError(f'{split} split is too small for the default 256-byte context. Add more reviewed data before training.')

## 2. Train DREAM from scratch

The starter settings use a small micro-batch plus gradient accumulation to reduce peak memory. Increase steps only if the runtime can finish. Training saves resumable checkpoints in the Colab session; download them before the session ends.

In [ ]:
TRAIN_STEPS = 2000
MICRO_BATCH = 4
ACCUMULATION = 8
subprocess.run([
    sys.executable, 'model/train.py',
    '--data', 'data/prepared/train.txt',
    '--validation-data', 'data/prepared/validation.txt',
    '--out', 'checkpoints/dream-small.pt',
    '--steps', str(TRAIN_STEPS),
    '--batch-size', str(MICRO_BATCH),
    '--grad-accum-steps', str(ACCUMULATION),
    '--block-size', '256', '--d-model', '256', '--layers', '6', '--heads', '8',
    '--eval-every', '100', '--checkpoint-every', '250', '--eval-batches', '10'
], check=True)

## 3. Evaluate on the held-out test split

This reports next-byte prediction metrics and a sample generation. These metrics do not prove reasoning, factual accuracy, safety or instruction-following ability.

In [ ]:
subprocess.run([
    sys.executable, 'model/evaluate.py',
    '--checkpoint', 'checkpoints/dream-small.pt',
    '--data', 'data/prepared/test.txt',
    '--batches', '50', '--batch-size', '8',
    '--out', 'eval-results/dream-small.json'
], check=True)
report = json.loads(Path('eval-results/dream-small.json').read_text())
print('Bits per byte:', report['bits_per_byte'])
print('Perplexity per byte:', report['perplexity_per_byte'])
print('Sample:', report['sample_generation'])

## 4. Optional: instruction-tune DREAM on reviewed conversations

This second stage starts from the DREAM checkpoint you just trained. It does not load another model. Provide separate `sft-train.jsonl` and `sft-validation.jsonl` files with rights/privacy/quality review completed. The script masks user/system prompts and learns only from assistant responses. Their source groups must be disjoint from each other and from the held-out pretraining test sources in `data/prepared/manifest.json`.

In [ ]:
uploaded_sft = files.upload()
required_sft = ('sft-train.jsonl', 'sft-validation.jsonl')
if any(name not in uploaded_sft for name in required_sft):
    raise RuntimeError('Upload both sft-train.jsonl and sft-validation.jsonl')
for name in required_sft:
    Path('data').mkdir(exist_ok=True)
    Path('data', name).write_bytes(uploaded_sft[name])
print('Training bytes:', Path('data/sft-train.jsonl').stat().st_size)
print('Validation bytes:', Path('data/sft-validation.jsonl').stat().st_size)

In [ ]:
SFT_STEPS = 1000
subprocess.run([
    sys.executable, 'model/sft.py',
    '--checkpoint', 'checkpoints/dream-small.pt',
    '--train-data', 'data/sft-train.jsonl',
    '--validation-data', 'data/sft-validation.jsonl',
    '--pretraining-manifest', 'data/prepared/manifest.json',
    '--out', 'checkpoints/dream-instruct.pt',
    '--steps', str(SFT_STEPS),
    '--batch-size', '2', '--grad-accum-steps', '4',
    '--eval-every', '100', '--checkpoint-every', '250',
    '--eval-batches', '10'
], check=True)
sft_completed = True

In [ ]:
subprocess.run([
    sys.executable, 'model/evaluate.py',
    '--checkpoint', 'checkpoints/dream-instruct.pt',
    '--data', 'data/prepared/test.txt',
    '--batches', '50', '--batch-size', '8',
    '--out', 'eval-results/dream-instruct.json'
], check=True)
sft_report = json.loads(Path('eval-results/dream-instruct.json').read_text())
print('Instruction checkpoint bits per byte:', sft_report['bits_per_byte'])
print('Sample:', sft_report['sample_generation'])

## 6. Download the trained DREAM checkpoints and reports

Download the pretraining and instruction-tuned checkpoints, both evaluation reports, and the dataset manifest. Preserve the exact reviewed datasets with these artifacts.

In [ ]:
files.download('checkpoints/dream-small.pt')
files.download('eval-results/dream-small.json')
files.download('data/prepared/manifest.json')
if sft_completed:
    files.download('checkpoints/dream-instruct.pt')
    files.download('eval-results/dream-instruct.json')
else:
    print('Instruction fine-tuning was skipped; only the pretraining checkpoint and report are downloaded.')